In [ ]:
import pulp
import numpy as np
import matplotlib.pyplot as plt
import os

# ==========================================
# 1. CONFIGURACIÓN Y GENERACIÓN DE DATOS
# ==========================================
np.random.seed(42) # Semilla para reproducibilidad
os.makedirs('figures', exist_ok=True) # Crear carpeta para las figuras

T = 20                 # Rondas de comunicación
num_edges = 3          # Servidores de borde (Edge Servers)
clients_per_edge = 10  # Clientes por servidor de borde
N = num_edges * clients_per_edge # Total de clientes (30)
K_min = 5              # Cuota mínima de clientes por ronda

# Mapeo de cliente a su respectivo Edge Server (0 a 9 -> Edge 0, etc.)
client_to_edge = {i: i // clients_per_edge for i in range(N)}

# Generación de datos sintéticos (Distribuciones uniformes)
B = np.random.uniform(500, 1000, N)           # Batería inicial (Joules)
E_cost = np.random.uniform(10, 30, (N, T))    # Consumo de energía por ronda (Joules)
CI = np.random.uniform(100, 500, (N, T))      # Intensidad de carbono (gCO2/kWh)
U = np.random.uniform(0.1, 1.0, (N, T))       # Utilidad estadística del modelo

# Definir un presupuesto de carbono H (80% del máximo posible para forzar optimización)
max_carbon_possible = np.sum(E_cost * CI)
H = 0.80 * max_carbon_possible



In [ ]:
# ==========================================
# 2. FORMULACIÓN DEL MODELO ILP (PuLP)
# ==========================================
prob = pulp.LpProblem("Optimal_Carbon_Aware_HFL", pulp.LpMaximize)

# Variables de Decisión Binarias
# y[e, t]: 1 si el Edge Server 'e' se activa en la ronda 't'
y = pulp.LpVariable.dicts("y", ((e, t) for e in range(num_edges) for t in range(T)), cat='Binary')
# x[i, t]: 1 si el cliente 'i' participa en la ronda 't'
x = pulp.LpVariable.dicts("x", ((i, t) for i in range(N) for t in range(T)), cat='Binary')

# Función Objetivo: Maximizar Utilidad Estadística
prob += pulp.lpSum(U[i, t] * x[i, t] for i in range(N) for t in range(T)), "Total_Utility"

# Restricciones
# 1. Topología Jerárquica: Un cliente solo participa si su Edge Server está activo
for t in range(T):
    for i in range(N):
        e = client_to_edge[i]
        prob += x[i, t] <= y[e, t], f"Topology_c{i}_t{t}"

# 2. Restricción de Batería Local
for i in range(N):
    prob += pulp.lpSum(E_cost[i, t] * x[i, t] for t in range(T)) <= B[i], f"Battery_c{i}"

# 3. Presupuesto de Carbono Operacional
prob += pulp.lpSum(E_cost[i, t] * CI[i, t] * x[i, t] for i in range(N) for t in range(T)) <= H, "Carbon_Budget"

# 4. Cuota Mínima de Participación
for t in range(T):
    prob += pulp.lpSum(x[i, t] for i in range(N)) >= K_min, f"Min_Quota_t{t}"



In [ ]:
# ==========================================
# 3. RESOLUCIÓN DEL MODELO
# ==========================================
print("Resolviendo el modelo con PuLP CBC...")
prob.solve(pulp.PULP_CBC_CMD(msg=False))
print(f"Estado de la solución: {pulp.LpStatus[prob.status]}")



Resolviendo el modelo con PuLP CBC...
Estado de la solución: Optimal


In [ ]:
# ==========================================
# 4. EXTRACCIÓN DE RESULTADOS Y GRÁFICAS
# ==========================================
if pulp.LpStatus[prob.status] == 'Optimal':

    # Extrayendo datos para las gráficas
    active_clients = np.zeros((num_edges, T))
    carbon_per_round = np.zeros(T)
    total_energy_consumed = 0

    for t in range(T):
        for i in range(N):
            if pulp.value(x[i, t]) == 1.0:
                e = client_to_edge[i]
                active_clients[e, t] += 1
                carbon_per_round[t] += (E_cost[i, t] * CI[i, t])
                total_energy_consumed += E_cost[i, t]

    cum_carbon = np.cumsum(carbon_per_round)

    # GRÁFICA 1: Topología (fig1_topology.png)
    plt.figure(figsize=(8, 5))
    bottom = np.zeros(T)
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    for e in range(num_edges):
        plt.bar(range(1, T+1), active_clients[e], bottom=bottom, color=colors[e], label=f'Edge Server {e+1}')
        bottom += active_clients[e]
    plt.axhline(y=K_min, color='red', linestyle='--', label=f'Min Quota ($K_{{min}}$)')
    plt.xlabel('Communication Round ($t$)')
    plt.ylabel('Number of Active Clients')
    plt.title('Client Participation per Edge Server')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('figures/fig1_topology.png', dpi=300)
    plt.close()

    # GRÁFICA 2: Huella de Carbono (fig2_carbon.png)
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, T+1), cum_carbon, marker='o', linestyle='-', color='teal', label='Cumulative Carbon')
    plt.axhline(y=H, color='red', linestyle='--', linewidth=2, label='Carbon Budget ($H$)')
    plt.fill_between(range(1, T+1), cum_carbon, color='teal', alpha=0.1)
    plt.xlabel('Communication Round ($t$)')
    plt.ylabel('Cumulative Carbon Emissions ($gCO_2eq$)')
    plt.title('Operational Carbon Footprint vs Budget')
    plt.legend()
    plt.grid(linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('figures/fig2_carbon.png', dpi=300)
    plt.close()

    # GRÁFICA 3: Batería (fig3_battery.png)
    plt.figure(figsize=(8, 5))
    sample_clients = np.random.choice(N, 5, replace=False) # 5 clientes al azar
    for idx, i in enumerate(sample_clients):
        battery_traj = [B[i]]
        for t in range(T):
            cost = E_cost[i, t] if pulp.value(x[i, t]) == 1.0 else 0
            battery_traj.append(battery_traj[-1] - cost)
        plt.plot(range(T+1), battery_traj, marker='.', label=f'Client {i} (Edge {client_to_edge[i]+1})')

    plt.xlabel('Communication Round ($t$)')
    plt.ylabel('Residual Battery (Joules)')
    plt.title('Battery Depletion Trajectory')
    plt.legend()
    plt.grid(linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('figures/fig3_battery.png', dpi=300)
    plt.close()

    # ==========================================
    # 5. DATOS PARA LA TABLA DE LATEX
    # ==========================================
    total_carbon_val = cum_carbon[-1]
    carbon_utilization = (total_carbon_val / H) * 100
    avg_active_edges = np.mean([sum(pulp.value(y[e, t]) for e in range(num_edges)) for t in range(T)])
    edge_utilization = (avg_active_edges / num_edges) * 100

    print("\n" + "="*50)
    print("DATOS PARA RELLENAR LA TABLA DE LATEX")
    print("="*50)
    print(f"Statistical Utility Achieved : {pulp.value(prob.objective):.2f}")
    print(f"Total Carbon Emissions       : {total_carbon_val:.2f} (Budget: {H:.2f}) -> {carbon_utilization:.1f}%")
    print(f"Total Energy Consumed        : {total_energy_consumed:.2f} Joules")
    print(f"Active Edge Servers (Avg)    : {avg_active_edges:.2f} -> {edge_utilization:.1f}% utilization")
    print("="*50)
    print("Las 3 imágenes han sido generadas y guardadas en la carpeta 'figures/'.")
else:
    print("El modelo no pudo encontrar una solución óptima. Revisa los parámetros.")


DATOS PARA RELLENAR LA TABLA DE LATEX
Statistical Utility Achieved : 310.00
Total Carbon Emissions       : 2877061.26 (Budget: 2877066.79) -> 100.0%
Total Energy Consumed        : 10117.24 Joules
Active Edge Servers (Avg)    : 3.00 -> 100.0% utilization
Las 3 imágenes han sido generadas y guardadas en la carpeta 'figures/'.
